# GNU Radio IIR Filtering for Synthetic IQ Signals

A reproducible tutorial that builds a type-correct GNU Radio flowgraph for low-pass filtering complex IQ samples.

## Audience, prerequisites, and learning goals

**Audience:** learners familiar with Python, NumPy, and basic IQ notation.

**Prerequisites:** GNU Radio Python bindings and NumPy. In this repository, create the virtual environment with `python3 -m venv --system-site-packages .venv` so it can see the apt-installed GNU Radio modules.

By the end, you will be able to:

1. Generate deterministic complex IQ samples.
2. Connect complex and float GNU Radio blocks without type mismatches.
3. Apply a recursive single-pole IIR filter independently to I and Q.
4. Verify output shape, finiteness, and smoothing behavior.

## Outline

1. Import and configure the experiment.
2. Generate a noisy complex sinusoid.
3. Define the GNU Radio flowgraph.
4. Run it and inspect signal statistics.
5. Execute focused assertions.
6. Try a short extension exercise.

In [ ]:
import numpy as np

try:
    from gnuradio import blocks, filter, gr
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "GNU Radio is required. Follow AGENTS.md and use a virtual environment "
        "created with --system-site-packages."
    ) from exc

## 1. Generate a synthetic IQ signal

The signal combines a low-frequency complex tone with complex Gaussian noise. A fixed seed makes every run reproducible.

In [ ]:
N = 5
L = 1_000
SEED = 42
ALPHA = 0.10

rng = np.random.default_rng(SEED)
t = np.arange(L, dtype=np.float32)
tone = np.exp(1j * 2 * np.pi * 0.02 * t)
noise = 0.35 * (
    rng.standard_normal((N, L)) + 1j * rng.standard_normal((N, L))
)
iq_signal = (tone[np.newaxis, :] + noise).astype(np.complex64)
iq_flat = iq_signal.reshape(-1)

print(f"IQ matrix shape: {iq_signal.shape}")
print(f"Flattened samples: {iq_flat.size}")
print(f"Input dtype: {iq_flat.dtype}")

## 2. Build a type-correct flowgraph

`vector_source_c` emits complex samples, but `filter.single_pole_iir_filter_ff` accepts floats and belongs to GNU Radio's `filter` module. Connecting them directly would be invalid. The flowgraph therefore splits complex samples into I and Q, filters both float streams independently, and recombines them before `vector_sink_c`.

```text
vector_source_c -> complex_to_float -> IIR(I) -> float_to_complex -> vector_sink_c
                                  \-> IIR(Q) -/
```

In [ ]:
class IIRFilterFlowgraph(gr.top_block):
    def __init__(self, samples, alpha=ALPHA):
        super().__init__("Complex IQ single-pole IIR filter")

        self.input_samples = np.asarray(samples, dtype=np.complex64).reshape(-1)
        self.source = blocks.vector_source_c(self.input_samples.tolist(), False)
        self.split_iq = blocks.complex_to_float(1)
        self.iir_i = filter.single_pole_iir_filter_ff(alpha, 1)
        self.iir_q = filter.single_pole_iir_filter_ff(alpha, 1)
        self.combine_iq = blocks.float_to_complex(1)
        self.sink = blocks.vector_sink_c()

        self.connect(self.source, self.split_iq)
        self.connect((self.split_iq, 0), self.iir_i, (self.combine_iq, 0))
        self.connect((self.split_iq, 1), self.iir_q, (self.combine_iq, 1))
        self.connect(self.combine_iq, self.sink)

    def output(self):
        return np.asarray(self.sink.data(), dtype=np.complex64)

## 3. Run the flowgraph and inspect statistics

The output is reshaped to the original `(N, L)` layout only after GNU Radio has processed the flat stream.

In [ ]:
flowgraph = IIRFilterFlowgraph(iq_flat)
flowgraph.run()
filtered_flat = flowgraph.output()
filtered_iq = filtered_flat.reshape(iq_signal.shape)

input_power = float(np.mean(np.abs(iq_signal) ** 2))
output_power = float(np.mean(np.abs(filtered_iq) ** 2))
input_step = float(np.mean(np.abs(np.diff(iq_flat))))
output_step = float(np.mean(np.abs(np.diff(filtered_flat))))

print(f"Input shape:  {iq_signal.shape}")
print(f"Output shape: {filtered_iq.shape}")
print(f"Input power:  {input_power:.6f}")
print(f"Output power: {output_power:.6f}")
print(f"Mean adjacent-sample change before: {input_step:.6f}")
print(f"Mean adjacent-sample change after:  {output_step:.6f}")

## 4. Focused verification

A low-pass IIR filter should preserve the sample count, produce finite values, alter the noisy input, and reduce average adjacent-sample variation.

In [ ]:
assert filtered_iq.shape == iq_signal.shape, (
    f"Expected output shape {iq_signal.shape}, got {filtered_iq.shape}"
)
assert np.isfinite(filtered_iq).all(), "Filtered output contains NaN or infinity"
assert not np.allclose(filtered_iq, iq_signal), "The filter did not change the signal"
assert output_step < input_step, (
    "Expected low-pass filtering to reduce adjacent-sample variation: "
    f"input={input_step:.6f}, output={output_step:.6f}"
)

print("PASS: output shape, finiteness, signal change, and smoothing verified.")

## Interpretation

The recursive filter computes a smoothed output from the current input and prior output. Applying identical filters to I and Q preserves the complex IQ representation while attenuating rapid sample-to-sample changes.

## Exercise

Change `ALPHA` to `0.02`, rebuild the flowgraph, and predict whether `output_step` will increase or decrease before running it.

**Answer scaffold:** A smaller alpha gives each new sample ______ influence, so the output should become ______ and `output_step` should ______.

In [ ]:
# Optional exercise
exercise_alpha = 0.02
# exercise_flowgraph = IIRFilterFlowgraph(iq_flat, alpha=exercise_alpha)
# exercise_flowgraph.run()
# exercise_output = exercise_flowgraph.output()
# exercise_step = float(np.mean(np.abs(np.diff(exercise_output))))
# print(f"Exercise mean adjacent-sample change: {exercise_step:.6f}")

## Common pitfall and extension

**Pitfall:** Do not connect `vector_source_c` directly to `single_pole_iir_filter_ff`; complex and float stream signatures are incompatible.

**Extension:** Plot the input and output spectra or compare several alpha values, while keeping the same deterministic input for a fair comparison.